
**Created by:**

__Viktor Varga__

<br>
<img src="../_files/nb_im/ik_logo.png" style="display:inline-block" width='40%'>
<hr>

# **NumPy Array Programming Guide** - Chapter 1


## The N-dimensional array: the `ndarray` type


This guide assumes a basic knowledge of Python.

_**Note:** For convenience and in line with standard practice, we import NumPy as `np` in our Python scripts. Whenever we refer to a NumPy operation or type below, we will abbreviate the NumPy package name as `np`. For example: `np.amax()` instead of `numpy.amax()`._


The `np.ndarray` type is the most essential type in NumPy. It is used for storing data, similarly to Python lists and to arrays found in other programming languages. Compared to Python's built-in `list` type it has many advantages, but we also lose some convenient properties of lists:

- Both a list (`[]`) and an `ndarray` point to a region of the memory. However, the memory region allocated for the list does not contain the actual elements of the list, but only stores references to them (i.e., their memory addresses), while the elements themselves are stored elsewhere in memory, as individual Python objects. Meanwhile, an array stored the values directly in the allocated memory region. (An exception is an array with `np.object` dtype\*.) These two implementation approaches imply different properties. Because a list stores only object references, its elements can be arbitrary, mixed-type objects, including other lists. **In contrast, the array's data type** (`ndarray.dtype`) **is homogeneous and fixed in advance.** For example, if the dtype is `np.int64`, then every element occupies 8 bytes (64 bits) in memory, and the array stores the numeric values directly.

- While a list can grow, the **array size is fixed at creation time**. If we want to add new elements, we must create a new, larger array and copy the old elements next to the new ones.

- Due to its implementation, an array is typically a much more efficient data structure than a list.

A special property of `ndarray` arrays is their apparent N-dimensional nature. The number of dimensions here should not be confused with the mathematical notion of a vector's dimension. In this context it refers to the number of axes along which we can arrange the elements. We distinguish between the **size** (`ndarray.size`) **and the shape** (`ndarray.shape`) of an array. The size tells how many elements the array contains. While the size of the array is fixed, we can reshape the array so that it appears to extend along multiple dimensions and we can index its elements not only by their linear position, but by where they lie along each axis of the multidimensional layout. In short, **we can reference elements using multidimensional indices**. The shape tells the extent of the array along each axis. This is extremely useful when working with data that is naturally indexable along multiple axes. For example, a color image can be represented as a 3D array whose shape is (image height, image width, number of color channels).

The figure below shows the apparent 2D layout and the in-memory layout of a 2D array with 6 elements, shape (2, 3), and a 4-byte dtype\*\*. The 2D structure is only apparent: in memory, the elements are stored sequentially.

<img src='../_files/nb_im/numpy_shape_vs_memory_en.png' width='60%'>

\* _**Note:** An array with `np.object` dtype stores references to arbitrary Python objects. Compared to arrays that store values directly, this approach is much less efficient. When possible, use primitive dtypes (`np.float32`, `np.int64`, `np.bool_`, etc.). An `np.object` array can be useful when you cannot express your data with primitive dtypes and you truly need to treat items as general Python objects, but you still want to benefit from some aspects of vectorized programming (for example indexing and broadcasting) that are not available with the basic Python container types._

\*\* _**Note:** An `ndarray` object does not necessarily reference a contiguous memory region. This will be discussed in the **basic indexing** and **views** section of [Chapter 2](numpy_tut24_chapter2_en.ipynb)._


## Creating arrays


Import NumPy and create a new array using `np.array()`!


In [ ]:
import numpy as np

a = np.array([[2,3,4],[1,5,8]], dtype=np.int32)

print("The values of the array:\n", a)

print("\na.shape (the shape of the array):", a.shape)   # a tuple
print("a.dtype (the data type of the array):", a.dtype)   # the type of the elements
print("a.ndim (the number of axes == the dimensions of the array):", a.ndim)   # a.k.a length of the shape tuple
print("a.size (the size of the array == total number of elements):", a.size)   # the product of the shape
print("a.shape[0] (the length along axis#0):", a.shape[0])
print("len(a) (the length along axis#0):", len(a))   # len() built-in python function


The values of the array:
 [[2 3 4]
 [1 5 8]]

a.shape (the shape of the array): (2, 3)
a.dtype (the data type of the array): int32
a.ndim (the number of axes == the dimensions of the array): 2
a.size (the size of the array == total number of elements): 6
a.shape[0] (the length along axis#0): 2
len(a) (the length along axis#0): 2


With `np.array()` we can create new arrays from existing data structures, possibly of other types. The function accepts lists and tuples, and also other NumPy arrays. We can also pass nested lists or tuples to obtain multidimensional arrays, as shown above. In that case, of course, every inner list/tuple at the same nesting level must have the same number of elements.

The shape of the created array is given by the array's `shape` field. This is a tuple that lists the lengths of the array's axes. The length of the `shape` tuple, that is the number of dimensions, is given by `ndim`. The `size` field stores the number of elements in the array, which is the product of the values in `shape` (overlapping window views are an exception, see `np.lib.stride_tricks.as_strided()`, `np.lib.stride_tricks.sliding_window_view()`). In the example above this is 2 x 3 = 6. `dtype` specifies the data type of the elements.

The `np.int32` dtype represents integers stored in 4 bytes; values are taken from the interval $[\ - 2^{31}, \  2^{31}-1\ ]$. Python's built-in `len()` function also works on arrays: it returns the length of the first (axis #0) dimension.


In [ ]:
zero_1d = np.array([], dtype=np.float32)
zero_2d = np.array([[]], dtype=np.float32)
one_3d = np.array([[[5]]], dtype=np.float32)

print("The shape of the first array:", zero_1d.shape)
print("The shape of the second array:", zero_2d.shape)
print("The shape of the third array:", one_3d.shape)

The shape of the first array: (0,)
The shape of the second array: (1, 0)
The shape of the third array: (1, 1, 1)


As you can see above, the depth of nested lists/tuples controls the shape of the resulting array.

Empty arrays can also be created. Even though they contain no elements, their shape and number of axes can still vary (in that case the product of axis lengths must be 0). Creating empty arrays is usually not very useful, but operations can produce them. For example, returning an array of shape (0, 480, 640) can conveniently signal that we intended to return 640 x 480 images, but for some reason we did not manage to produce any.


In [ ]:
b_i32 = np.array([-2,3,400], dtype=np.int32)
b_fl32 = np.array([-2,3,400], dtype=np.float32)
b_bool = np.array([True,False,True], dtype=np.bool_)

print("The values of the int32 array:", b_i32)
print("The values of the float32 array:", b_fl32)
print("The values of the boolean array:", b_bool)

The values of the int32 array: [ -2   3 400]
The values of the float32 array: [ -2.   3. 400.]
The values of the boolean array: [ True False  True]


An array can have many different dtypes. The most commonly used ones are:
- `np.int32` and `np.int64` for 4- and 8-byte signed integers.
- `np.float32` and `np.float64` for real numbers (neural networks typically operate on these).
- `np.uint8` for 1-byte unsigned integers (image pixel intensities are often stored in 8 bits using values in [0, 255]).
- `np.bool_` for booleans (it can take `True` and `False`, but it uses 1 byte rather than 1 bit, because the smallest individually addressable unit of memory is a byte).

It is usually worth specifying the dtype explicitly when creating arrays. If you are not careful, underflow or overflow can easily occur, as in the example below.


In [ ]:
b_ui8 = np.array([-2,3,400], dtype=np.uint8)   # raises Warning for under/overflow after NumPy version 1.24.

print("The values of the uint8 array:", b_ui8)

The values of the uint8 array: [254   3 144]


<ipython-input-6-009dbf8f8a7e>:1: DeprecationWarning: NumPy will stop allowing conversion of out-of-bound Python integers to integer arrays.  The conversion of -2 to uint8 will fail in the future.
For the old behavior, usually:
    np.array(value).astype(dtype)
will give the desired result (the cast overflows).
  b_ui8 = np.array([-2,3,400], dtype=np.uint8)   # throws Warning for under/overflow after NumPy version 1.24.
<ipython-input-6-009dbf8f8a7e>:1: DeprecationWarning: NumPy will stop allowing conversion of out-of-bound Python integers to integer arrays.  The conversion of 400 to uint8 will fail in the future.
For the old behavior, usually:
    np.array(value).astype(dtype)
will give the desired result (the cast overflows).
  b_ui8 = np.array([-2,3,400], dtype=np.uint8)   # throws Warning for under/overflow after NumPy version 1.24.


Sometimes you may want to change an array's dtype later, after initialization. This often happens when some operation expects a specific dtype for its input arrays, or returns an array with a specific dtype that does not match what other operations expect. For example, the element-wise absolute value operation `np.fabs()` always returns a floating-point array, even if it receives an integer array.

In such cases you can create a new array with the same shape as the original but a different dtype using `ndarray.astype()`. The function copies the elements of the input array into a new array after converting them one by one to the requested dtype:


In [ ]:
a = np.array([[-2,3,-4],[-1,-5,8]], dtype=np.int32)

print("Data type (dtype) of the 'a' array:", a.dtype)
print("Contents of the 'a' array:\n", a)

abs_a = np.fabs(a)
print("\nData type (dtype) of the 'np.fabs(a)' array:", abs_a.dtype)
print("Contents of the 'np.fabs(a)' array:\n", abs_a)

abs_a_i32 = abs_a.astype(np.int32)
print("\nData type (dtype) of the 'np.fabs(a).astype(np.int32)' array:", abs_a_i32.dtype)
print("Contents of the 'np.fabs(a).astype(np.int32)' array:\n", abs_a_i32)

Data type (dtype) of the 'a' array: int32
Contents of the 'a' array:
 [[-2  3 -4]
 [-1 -5  8]]

Data type (dtype) of the 'np.fabs(a)' array: float64
Contents of the 'np.fabs(a)' array:
 [[2. 3. 4.]
 [1. 5. 8.]]

Data type (dtype) of the 'np.fabs(a).astype(np.int32)' array: int32
Contents of the 'np.fabs(a).astype(np.int32)' array:
 [[2 3 4]
 [1 5 8]]


However, it is not necessary to initialize arrays only from lists or tuples. We can also create arrays filled with zeros or ones using `np.zeros()` and `np.ones()`. Here the first parameter is the desired shape.

Zeros and ones play a special role for boolean values, because 0 and 1 correspond to `False` and `True` under the `bool_` interpretation. An `np.bool_` array created with `np.ones()` therefore contains only `True` values.


In [ ]:
zeros1 = np.zeros((2,3), dtype=np.int32)
zeros2 = np.zeros((3,), dtype=np.float32)
zeros3 = np.zeros((3,), dtype=np.bool_)
ones1 = np.ones((2, 3), dtype=np.int32)
ones2 = np.ones((3,), dtype=np.float32)
ones3 = np.ones((3,), dtype=np.bool_)

print("zeros1.shape:", zeros1.shape)
print("ones2.shape:", ones2.shape)

print("\nzeros1 (int):\n", zeros1)
print("\nzeros2 (float):", zeros2)
print("zeros3 (bool):", zeros3)

print("\nones1 (int):\n", ones1)
print("\nones2 (float):", ones2)
print("ones3 (bool):", ones3)

zeros1.shape: (2, 3)
ones2.shape: (3,)

zeros1 (int):
 [[0 0 0]
 [0 0 0]]

zeros2 (float): [0. 0. 0.]
zeros3 (bool): [False False False]

ones1 (int):
 [[1 1 1]
 [1 1 1]]

ones2 (float): [1. 1. 1.]
ones3 (bool): [ True  True  True]


In addition to ones and zeros, `np.full()` allows you to create new arrays with other initial values as well:


In [ ]:
full1 = np.full((3,), dtype=np.float32, fill_value=2.5)
full2 = np.full((3,), dtype=np.float32, fill_value=[1,2,3])
full3 = np.full((3,), dtype=np.bool_, fill_value=True)

print("full1 (float):", full1)
print("full2 (float):", full2)
print("full3 (bool):", full3)

full1 (float): [2.5 2.5 2.5]
full2 (float): [1. 2. 3.]
full3 (bool): [ True  True  True]


We can create copies of our arrays using `np.copy()`. We can also create arrays filled with specific values whose shape and dtype mimic another array. The functions `np.zeros_like()`, `np.ones_like()`, and `np.full_like()` create new arrays with a similar shape and/or dtype to the given array, filled with the requested values.


In [ ]:
original_arr = np.full((3,), dtype=np.int32, fill_value=-1)

arr_copy = np.copy(original_arr)
arr_zeros = np.zeros_like(original_arr)
arr_ones = np.ones_like(original_arr)
arr_full = np.full_like(original_arr, fill_value=-2)
arr_ones_bool = np.ones_like(original_arr, dtype=np.bool_)

print("original_arr:", original_arr)
print("copy:", arr_copy)
print("zeros_like original_arr:", arr_zeros)
print("ones_like original_arr:", arr_ones)
print("full_like original_arr filled with -2:", arr_full)
print("ones_like original_arr with bool_ dtype:", arr_ones_bool)

original_arr: [-1 -1 -1]
copy: [-1 -1 -1]
zeros_like original_arr: [0 0 0]
ones_like original_arr: [1 1 1]
full_like original_arr filled with -2: [-2 -2 -2]
ones_like original_arr with bool_ dtype: [ True  True  True]


`np.arange()` is similar to Python's built-in `range()`: it produces values from an interval of integers, not necessarily with a step size of 1. The difference is that while `range()` returns a generator-like object from which values are produced lazily, on demand only, `np.arange()` eagerly creates all values and stores them in the returned array.


In [ ]:
my_range1 = np.arange(5)
my_range2 = np.arange(2,10)
my_range3 = np.arange(10,2)
my_range4 = np.arange(-10,2)
my_range5 = np.arange(-10,2,2)
my_range6 = np.arange(10,2,-1)
my_range7 = np.arange(2,10,-2)
my_range8 = np.arange(10,2,-1, dtype=np.float32)

print("arange(5):", my_range1)
print("arange(2,10):", my_range2)
print("arange(10,2):", my_range3)
print("arange(-10,2):", my_range4)
print("arange(-10,2,2):", my_range5)
print("arange(10,2,-1):", my_range6)
print("arange(2,10,-2):", my_range7)
print("arange(10,2,-1) float:", my_range8)

arange(5): [0 1 2 3 4]
arange(2,10): [2 3 4 5 6 7 8 9]
arange(10,2): []
arange(-10,2): [-10  -9  -8  -7  -6  -5  -4  -3  -2  -1   0   1]
arange(-10,2,2): [-10  -8  -6  -4  -2   0]
arange(10,2,-1): [10  9  8  7  6  5  4  3]
arange(2,10,-2): []
arange(10,2,-1) float: [10.  9.  8.  7.  6.  5.  4.  3.]


`np.arange()`, like `range()`, can be called with at least one and at most three numeric parameters:

*(start, (exclusive) stop, step)*

If *start* >= *stop* and *step* is not negative, you will get an empty array. A negative step is also allowed, in which case the interval is traversed backwards.

As with `range()`, the function can be called with one or two parameters. With two parameters you do not specify the step, so it defaults to 1. With only one parameter you specify the stop value and the start defaults to 0. Besides these parameters, you can also provide `dtype`, among others, which determines the dtype of the returned array.

`np.arange()` will play an important role in *advanced indexing* (see [Chapter 4](numpy_tut24_chapter4_en.ipynb)).


## Accessing individual array elements


One of the key characteristics of vectorized programming is that, with appropriate indexing, we can refer to multiple elements at once without having to use loops or similar syntactic constructs.

In this chapter we will only look at the simplest form of indexing multidimensional arrays: **referencing a single element**.


In [ ]:
a = np.array([[2,3,4],[1,5,8]], dtype=np.int32)

print("a[0,0]:", a[0,0])
print("a[1,0]:", a[1,0])
print("a[0,2]:", a[0,1])
print("a[1,2]:", a[1,2])
#print("a[3,2] raises error:", a[3,2]) # -> IndexError: index 3 is out of bounds for axis 0 with size 2

print("a[-1,0]:", a[-1,0])     # a[-1,0] == a[1,0]
print("a[0,-1]:", a[0,-1])     # a[0,-1] == a[0,2]
print("a[-1,-1]:", a[-1,-1])   # a[-1,-1] == a[1,2]

a[0,0]: 2
a[1,0]: 1
a[0,2]: 3
a[1,2]: 8
a[-1,0]: 1
a[0,-1]: 4
a[-1,-1]: 8


As you can see above, an element of a two-dimensional array can be referenced using `[idx0, idx1]`, where `idx0` is the index along axis #0 and `idx1` is the index along axis #1.

As with Python data structures, negative indices do not raise an error. Instead, the index `-idx` (along the given axis `i`) refers to element `shape[i]-idx`, that is the `idx`-th element counted from the end. For example, `[-1, -1]` refers to the last element of a 2D array.

Elements referenced in this way are also writable:


In [ ]:
a = np.array([[2,3,4],[1,5,8]], dtype=np.int32)
a[0,0] = 42
a[-1,0] = 99
b = np.copy(a)
a[0,-1] = -99
a[-1,-1] = 55

print("The modified array:\n", a)
print("\nThe array copied halfway thorugh:\n", b)

The modified array:
 [[ 42   3 -99]
 [ 99   5  55]]

The array copied halfway thorugh:
 [[42  3  4]
 [99  5  8]]


## Simple element-wise operations


Element-wise operations are one of the main groups of functions available in NumPy. **These operations are originally defined for simple scalars, but in NumPy they are extended to whole arrays.** A typical example is addition. Adding two arrays of the same shape element by element can be decomposed into as many independent scalar additions as there are elements. Such vectorized operations are massively parallelizable for large arrays. Although NumPy parallelizes only a subset of operations through BLAS and LAPACK, TensorFlow and PyTorch automatically parallelize most element-wise operations on GPUs.

Let us define two arrays of the same shape (`(2, 3)`) containing numbers!


In [ ]:
arr1 = np.array([[2,3,4],[1,5,8]], dtype=np.int32)
arr2 = np.array([[5,3,1],[6,4,-2]], dtype=np.int32)

First, let us see how we would add the corresponding elements of two arrays using traditional loops in Python!


In [ ]:
arr_result = np.zeros_like(arr1)
for idx0 in range(arr1.shape[0]):
  for idx1 in range(arr1.shape[1]):
    arr_result[idx0, idx1] = arr1[idx0, idx1] + arr2[idx0, idx1]

print("arr1 + arr2 is:\n", arr_result)

arr1 + arr2 is:
 [[7 6 5]
 [7 9 6]]


Above we had to use loops, because Python's addition operator (`+`) is defined only for scalars. NumPy's `+` operator extends this to arrays of arbitrary shape and dimensionality, **so the loops can be omitted**:


In [ ]:
arr_result = arr1 + arr2
print("arr1 + arr2 is:\n", arr_result)

arr_result = np.add(arr1, arr2)    # the '+' NumPy operator is equivalent to the np.add() function
print("\narr1 + arr2 is:\n", arr_result)

arr1 + arr2 is:
 [[7 6 5]
 [7 9 6]]

arr1 + arr2 is:
 [[7 6 5]
 [7 9 6]]


NumPy's `+` operator is the infix form of `np.add()`. We can also add arrays and scalar values:


In [ ]:
arr_result = arr1 + 42
print("arr1 + 42 is:\n", arr_result)

arr1 + 42 is:
 [[44 45 46]
 [43 47 50]]


The example above shows that NumPy's `+` operator is invoked even if only one operand is an `ndarray` (the other here is a Python `int`). When we add an array and a scalar, NumPy wraps the scalar into an array of shape (1,), then extends its shape to match the shape of the other array operand via virtual repetitions. So in practice this is still an element-wise addition of two arrays. The shape matching is performed by the so-called *broadcasting* mechanism, which we will discuss in more detail in later chapters.

Besides addition, many unary, binary, and multi-argument element-wise operations are available. A few of them are listed below:


In [ ]:
# numeric x numeric -> numeric
print("arr1 - arr2 is:\n", arr1 - arr2)   # np.subtract()
print("\narr1 * arr2 is:\n", arr1 * arr2)   # np.multiply()
print("\narr1 / (true div) arr2 is:\n", arr1 / arr2)   # np.true_divide()
print("\narr1 // (floor div) arr2 is:\n", arr1 // arr2)   # np.floor_divide()
print("\narr1 % (mod) arr2 is:\n", arr1 % arr2)   # np.mod()
print("\narr1 ** (power) arr2 is:\n", arr1 ** arr2.astype(np.float32))   # np.power()

print("\nmaximum(arr1, arr2) is:\n", np.maximum(arr1, arr2))
print("\nminimum(arr1, arr2) is:\n", np.minimum(arr1, arr2))

# numeric -> numeric
print("\n-arr1 is:\n", -arr1)   # np.negative()
print("\nabs(arr1) is:\n", np.fabs(arr1))


arr1 - arr2 is:
 [[-3  0  3]
 [-5  1 10]]

arr1 * arr2 is:
 [[ 10   9   4]
 [  6  20 -16]]

arr1 / (true div) arr2 is:
 [[ 0.4         1.          4.        ]
 [ 0.16666667  1.25       -4.        ]]

arr1 // (floor div) arr2 is:
 [[ 0  1  4]
 [ 0  1 -4]]

arr1 % (mod) arr2 is:
 [[2 0 0]
 [1 1 0]]

arr1 ** (power) arr2 is:
 [[3.2000e+01 2.7000e+01 4.0000e+00]
 [1.0000e+00 6.2500e+02 1.5625e-02]]

maximum(arr1, arr2) is:
 [[5 3 4]
 [6 5 8]]

minimum(arr1, arr2) is:
 [[ 2  3  1]
 [ 1  4 -2]]

-arr1 is:
 [[-2 -3 -4]
 [-1 -5 -8]]

abs(arr1) is:
 [[2. 3. 4.]
 [1. 5. 8.]]


As you can see, there are two kinds of division. `/` (true divide) returns exact values in a floating-point array, while `//` (floor divide) performs integer floor division.

`np.maximum()` writes the larger of the corresponding elements of two arrays into the result array. `np.minimum()` works analogously.

Of course, not only for `+` but for any similar operation you can also provide a scalar as one of the arguments. For example, `np.maximum(arr, 0)` returns a copy of `arr` in which all negative elements are replaced by zeros.

In TensorFlow and PyTorch these same operations are almost always available as well, typically under similar names (for example `tensorflow.math.maximum()`, `torch.maximum()`). We will learn about further element-wise operations later.


Element-wise NumPy operations can also be used *in-place*: in that case the result is written into one of the argument arrays. This can be more efficient in some cases, because it avoids allocating one additional array.

Sticking with addition, the in-place operator form is `+=`. In the following two lines we first add 2 to every element of the array on the left, and then add the array to itself:


In [ ]:
print("arr1:\n", arr1)

arr1 += 2
print("\narr1 after in-place increment by 2:\n", arr1)

arr1 += arr1
print("\narr1 after in-place increment by its own values:\n", arr1)

arr1:
 [[30 34 38]
 [26 42 54]]

arr1 after in-place increment by 2:
 [[32 36 40]
 [28 44 56]]

arr1 after in-place increment by its own values:
 [[ 64  72  80]
 [ 56  88 112]]


The in-place variant of addition can also be performed by calling `np.add()` directly. If you provide an array via the `out` parameter, the result is written there. `np.add(arr1, 2, out=arr1)` is equivalent to `arr1 += 2`, but the former also returns the array passed via `out`. Using `out` is especially useful for in-place computation when an operation has no operator form (for example `np.maximum()`).


In [ ]:
print("arr1:\n", arr1)

np.add(arr1, 2, out=arr1)
print("\narr1 after in-place increment by 2:\n", arr1)

arr2 = np.add(arr1, 2, out=arr1)
print("\narr1 after in-place increment again by 2:\n", arr1)
print("arr1 and arr2 variables are pointing to the same array object:", arr1 is arr2)

arr1:
 [[ 72  80  88]
 [ 64  96 120]]

arr1 after in-place increment by 2:
 [[ 74  82  90]
 [ 66  98 122]]

arr1 after in-place increment again by 2:
 [[ 76  84  92]
 [ 68 100 124]]
arr1 and arr2 variables are pointing to the same array object: True


## Simple _vector $\rightarrow$ scalar_ (collapse) operations


Another important group of NumPy operations are the *vector $\rightarrow$ scalar* operations, that are sometimes referred to as collapse operations. **Their input is an array and they produce a single value.** The vector-to-scalar analogue of the addition operation is summation.

Let us define an array of shape `(2, 3)` containing numbers!


In [ ]:
arr3 = np.array([[2,3,4],[1,5,8]], dtype=np.int32)

Let's implement the summation of the array elements first using a loop in Python!


In [ ]:
sum_ = 0
for idx0 in range(arr3.shape[0]):
  for idx1 in range(arr3.shape[1]):
    sum_ += arr3[idx0, idx1]

print("Sum of arr3 elements is:", sum_)

Sum of arr3 elements is: 23


Summation is available in NumPy as a standalone function. Our solution will be much more efficient if we use it:


In [ ]:
sum_ = np.sum(arr3)    # Note: do not use 'sum', 'max', 'min', etc. in Python as
                       #       variable names as these are built-in functions
                       #       and can be hidden by a redefinition of their names

print("Sum of arr3 elements is:", sum_)

Sum of arr3 elements is: 23


A few additional similar operations are listed below:


In [ ]:
print("np.sum(arr3):", np.sum(arr3))
print("np.prod(arr3):", np.prod(arr3))   # product of elements
print("np.mean(arr3):", np.mean(arr3))
print("np.std(arr3):", np.std(arr3))   # standard deviation (szoras)
print("np.var(arr3):", np.var(arr3))   # variance (szorasnegyzet)
print("np.amax(arr3):", np.amax(arr3))
print("np.amin(arr3):", np.amin(arr3))

np.sum(arr3): 23
np.prod(arr3): 960
np.mean(arr3): 3.8333333333333335
np.std(arr3): 2.266911751455907
np.var(arr3): 5.138888888888888
np.amax(arr3): 8
np.amin(arr3): 1


`np.amax()` and `np.amin()` return the maximum and minimum element of an array. Do not confuse them with the element-wise maximum/minimum functions `np.maximum()` and `np.minimum()` discussed above.
